# GeoGebra applet interaction from Python

In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [2]:
import ggblab, inspect
print('ggblab module file:', inspect.getsourcefile(ggblab))
from ggblab import comm
print('comm file:', getattr(comm, '__file__', None))
from ggblab.comm import ggb_comm_instance
print('has flush:', hasattr(ggb_comm_instance, '_flush_outbound_queue'))
print('methods:', [n for n in dir(ggb_comm_instance) if n.startswith('_flush')])

SyntaxError: expected 'except' or 'finally' block (comm.py, line 646)

In [3]:
from ggblab.comm import ggb_comm_instance

# ensure OOB server running
try:
    ggb_comm_instance.start()
except Exception:
    pass

# create a Comm (new API if available, fallback to ipykernel)
try:
    from comm import create_comm
    c = create_comm('ggblab-comm')
except Exception:
    from ipykernel.comm import Comm
    c = Comm(target_name='ggblab-comm')

# adopt it so ggblab uses this comm immediately
ggb_comm_instance.adopt_comm(c)

# now send
ggb_comm_instance.send({'type':'kernel-test','payload':'hello-frontend'})
print('sent, id:', getattr(c, 'comm_id', None) or getattr(c, 'commId', None))

sent, id: 419a21383d794d3bbf1d3f56f475bf29


In [19]:
import time
from ggblab.comm import ggb_comm_instance

def wait_for_applet(g, timeout=10.0):
    waited = 0.0
    interval = 0.1
    while waited < timeout:
        st = g.status()
        # has_oob_clients / target_comm_id / _oob_ready のいずれかで準備判定
        if st.get('has_oob_clients') or st.get('target_comm_id') or getattr(g, '_oob_ready', False):
            return True
        time.sleep(interval)
        waited += interval
    return False

if wait_for_applet(ggb_comm_instance, timeout=10.0):
    ggb_comm_instance.send({'type':'kernel-test','payload':'hello-frontend'})
else:
    print('Applet not ready after timeout; try later.')

In [1]:
# カーネルで実行（ノートブック）
from ggblab.comm import ggb_comm_instance

# 1) OOB サーバを起動（既に起動済みなら noop）
ggb_comm_instance.start()

# 2) カーネル側で Comm を作成して採用
try:
    from comm import create_comm
    c = create_comm('ggblab-comm')
except Exception:
    from ipykernel.comm import Comm
    c = Comm(target_name='ggblab-comm')

ggb_comm_instance.adopt_comm(c)

# 3) キューをフラッシュ（内部 API を呼ぶがデバッグ用）
ggb_comm_instance._flush_outbound_queue()

# 4) 状態確認
print(ggb_comm_instance.status())

SyntaxError: expected 'except' or 'finally' block (comm.py, line 646)

In [7]:
from ggblab import comm
g = comm.ggb_comm_instance
g.send({'type':'function','payload':'getVersion_via_ggb'})
print('ggb send invoked')

RuntimeError: No active Comm: Comm target not registered. Call ggb_comm_instance.register_target(<name>) in the kernel or ensure the frontend requests registration before sending.

In [6]:
ggb.comm.status()

{'target_name': 'ggblab-comm',
 'registered': True,
 'target_comm_id': None,
 'has_oob_clients': False,
 'socketPath': None,
 'wsPort': 0}

In [8]:
km = get_ipython().kernel.comm_manager
for cid, c in list(km.comms.items()):
    print(cid, getattr(c, 'target_name', None), type(c))

In [2]:
import xml.etree.ElementTree as ET
import xmlschema
import base64
import asyncio

In [3]:
from ggblab import GeoGebra

In [4]:
ggb = GeoGebra()

Using local cached file: xsd/common.xsd


In [5]:
await ggb.init()

Repointed ggb.comm -> ggb_comm_instance


In [13]:
await ggb.function("getVersion")

TimeoutError in send_recv {'type': 'function', 'payload': {'name': 'getVersion', 'args': None}, 'id': '59ca40c0-0c4a-4499-8af2-d9b650eef1e8'}


TimeoutError: oob future timed out

In [8]:
ggb.comm.recv_events.queue

deque([])

In [12]:
# カーネルで実行（ノートブック）
from ggblab.comm import ggb_comm_instance

# 1) OOB サーバを起動（既に起動済みなら noop）
ggb_comm_instance.start()

# 2) カーネル側で Comm を作成して採用
try:
    from comm import create_comm
    c = create_comm('ggblab-comm')
except Exception:
    from ipykernel.comm import Comm
    c = Comm(target_name='ggblab-comm')

ggb_comm_instance.adopt_comm(c)

# 3) キューをフラッシュ（内部 API を呼ぶがデバッグ用）
ggb_comm_instance._flush_outbound_queue()

# 4) 状態確認
print(ggb_comm_instance.status())

{'target_name': 'ggblab-comm', 'registered': True, 'target_comm_id': '548fdb42a3a04a7598ea565ddf0b1db4', 'has_oob_clients': False, 'socketPath': None, 'wsPort': 0}


In [27]:
from ggblab import comm
g = comm.ggb_comm_instance
print('status:', g.status())
tc = getattr(g, 'target_comm', None)
print('target_comm:', tc)
print('target_comm.comm_id:', getattr(tc, 'comm_id', None))
# direct send on the underlying Comm
tc.send({'type':'function','payload':'getVersion_direct'})
print('direct send done')

status: {'target_name': 'ggblab-comm', 'registered': True, 'target_comm_id': 'f91b8a085621478f8bdc572eaa6da38c', 'has_oob_clients': False, 'socketPath': '/tmp/ggb_h2n5qkeg', 'wsPort': 0}
target_comm: <ipykernel.comm.comm.BaseComm object at 0x129dd8f50>
target_comm.comm_id: f91b8a085621478f8bdc572eaa6da38c
direct send done


In [25]:
g.send({'type':'function','payload':'getVersion_via_ggb'})
print('ggb_comm.send called')

ggb_comm.send called


In [26]:
kernel = get_ipython().kernel
io_loop = getattr(kernel, 'io_loop', None)
print('io_loop:', io_loop)
if io_loop is not None and hasattr(io_loop, 'add_callback'):
    io_loop.add_callback(lambda: tc.send({'type':'function','payload':'getVersion_scheduled'}))
    print('scheduled send via io_loop')
else:
    print('no io_loop.add_callback available')

io_loop: <tornado.platform.asyncio.AsyncIOMainLoop object at 0x1196aaf90>
scheduled send via io_loop


In [3]:
import ggblab, inspect
print('ggblab path:', inspect.getsourcefile(ggblab))
import ggblab.comm as comm
print('comm path:', getattr(comm, '__file__', None))

ggblab path: /Users/manabu/miniforge3/envs/py314/lib/python3.14/site-packages/ggblab/__init__.py
comm path: /Users/manabu/miniforge3/envs/py314/lib/python3.14/site-packages/ggblab/comm.py


In [4]:
from ggblab import GeoGebra
print('import ok')

from ggblab.comm import ggb_comm_instance
print('ggb_comm_instance:', type(ggb_comm_instance))

import ok
ggb_comm_instance: <class 'ggblab.comm.ggb_comm'>


In [1]:
from ggblab import GeoGebra

In [5]:
await ggb.function("getVersion")

RuntimeError: No active Comm after waiting for frontend; ensure the frontend opened the Comm target and retry.

In [2]:
# open GeoGebra Widget on left-side
# ggb = await GeoGebra().init()

In [3]:
# ggb.comm.recv_events.queue
# ggb.comm.logs

In [14]:
# 再度インポート確認
from ggblab import GeoGebra  # または: from ggblab.comm import ggb_comm_instance
print('import ok')

# 送受信テスト（新旧 API に対応）
try:
    from comm import create_comm
    c = create_comm('ggblab-comm')
except Exception:
    from ipykernel.comm import Comm
    c = Comm(target_name='ggblab-comm')

from ggblab.comm import ggb_comm_instance
ggb_comm_instance.adopt_comm(c)
c.send({'type':'kernel-test','payload':'hello-frontend'})
print('sent, id:', getattr(c, 'comm_id', None))

import ok
sent, id: f91b8a085621478f8bdc572eaa6da38c


In [15]:
from ipykernel.comm import Comm
c = Comm(target_name='ggblab-comm')
c.send({'type':'function','payload':'getVersion'})
print('sent, id:', getattr(c,'comm_id',None))

sent, id: aa9261cc18144440a38f93f72da3cf2f


/var/folders/s8/67tp1kjd0pq2lymz2y_34fg00000gn/T/ipykernel_41466/690634144.py:2: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  c = Comm(target_name='ggblab-comm')


In [20]:
ggb.comm.send({'type':'function','payload':'getVersion'})

In [17]:
ggb.comm.recv_events.queue

deque([])

In [19]:
ggb.comm.logs

['No clients; waiting for client before sending bd2c6b7c-2382-4867-ba85-05d2047de3ba',
 'No clients; waiting for client before sending dfc8c413-d074-480d-92b9-b3d10bef6994',
 'No clients; waiting for client before sending 7a433f72-592c-4487-8170-d0ba9b149f83',
 'No clients; waiting for client before sending 59ca40c0-0c4a-4499-8af2-d9b650eef1e8']

In [21]:
from ggblab import comm
g = comm.ggb_comm_instance
print('status:', g.status())
print('target_comm:', getattr(g, 'target_comm', None))
print('outbound_queue_len:', len(getattr(g, '_outbound_queue', [])))

status: {'target_name': 'ggblab-comm', 'registered': True, 'target_comm_id': 'f91b8a085621478f8bdc572eaa6da38c', 'has_oob_clients': False, 'socketPath': '/tmp/ggb_h2n5qkeg', 'wsPort': 0}
target_comm: <ipykernel.comm.comm.BaseComm object at 0x129dd8f50>
outbound_queue_len: 0


In [22]:
g.send({'type':'function','payload':'getVersion'})
print('send() called')

send() called


In [23]:
from ipykernel.comm import Comm
c = Comm(target_name='ggblab-comm')
c.send({'type':'function','payload':'getVersion'})
print('direct comm sent id:', getattr(c, 'comm_id', None))

direct comm sent id: 93a3f65de12f47108412353968505ddb


/var/folders/s8/67tp1kjd0pq2lymz2y_34fg00000gn/T/ipykernel_41466/3426314478.py:2: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  c = Comm(target_name='ggblab-comm')


In [8]:
# in notebook
try:
    from comm import create_comm
    c = create_comm('ggblab-comm')
except Exception:
    from ipykernel.comm import Comm
    c = Comm(target_name='ggblab-comm')

from ggblab.comm import ggb_comm_instance
ggb_comm_instance.adopt_comm(c)

In [2]:
ggb = GeoGebra()

Using local cached file: xsd/common.xsd


In [3]:
await ggb.init()

Repointed ggb.comm -> ggb_comm_instance


In [12]:
ggb.comm.send({'type':'kernel-test','payload':'hello-frontend'})

RuntimeError: No active Comm: target registered but comm_open not received yet. Ensure the frontend has created the Comm for the requested target and retry.

In [10]:
await ggb.function("getVersion")

RuntimeError: No active Comm after waiting for frontend; ensure the frontend opened the Comm target and retry.

In [6]:
import pprint
pprint.pprint({
    'has_ggb': 'ggb' in globals(),
    'has_inst': 'ggb_comm_instance' in globals(),
    'ggb_obj': globals().get('ggb'),
    'ggb_comm_attr': getattr(globals().get('ggb'), 'comm', None),
    'ggb_comm_instance': globals().get('ggb_comm_instance'),
    'ids': {
        'id(ggb)': id(globals().get('ggb')) if 'ggb' in globals() else None,
        'id(ggb.comm)': id(getattr(globals().get('ggb'), 'comm', None)) if 'ggb' in globals() else None,
        'id(ggb_comm_instance)': id(globals().get('ggb_comm_instance')) if 'ggb_comm_instance' in globals() else None
    }
})

{'ggb_comm_attr': None,
 'ggb_comm_instance': None,
 'ggb_obj': <ggblab.ggbapplet.GeoGebra object at 0x1059afe00>,
 'has_ggb': True,
 'has_inst': False,
 'ids': {'id(ggb)': 4389010944,
         'id(ggb.comm)': 4311692656,
         'id(ggb_comm_instance)': None}}


In [7]:
t = asyncio.ensure_future(ggb.init())
await asyncio.sleep(1)
await t
await asyncio.sleep(1)
await ggb.function("getVersion")

Repointed ggb.comm -> ggb_comm_instance


RuntimeError: No active Comm: target registered but comm_open not received yet. Ensure the frontend has created the Comm for the requested target and retry.

In [8]:
import pprint
pprint.pprint({
    'has_ggb': 'ggb' in globals(),
    'has_inst': 'ggb_comm_instance' in globals(),
    'ggb_obj': globals().get('ggb'),
    'ggb_comm_attr': getattr(globals().get('ggb'), 'comm', None),
    'ggb_comm_instance': globals().get('ggb_comm_instance'),
    'ids': {
        'id(ggb)': id(globals().get('ggb')) if 'ggb' in globals() else None,
        'id(ggb.comm)': id(getattr(globals().get('ggb'), 'comm', None)) if 'ggb' in globals() else None,
        'id(ggb_comm_instance)': id(globals().get('ggb_comm_instance')) if 'ggb_comm_instance' in globals() else None
    }
})

{'ggb_comm_attr': <ggblab.comm.ggb_comm object at 0x108e05be0>,
 'ggb_comm_instance': <ggblab.comm.ggb_comm object at 0x108e05be0>,
 'ggb_obj': <ggblab.ggbapplet.GeoGebra object at 0x1059afe00>,
 'has_ggb': True,
 'has_inst': True,
 'ids': {'id(ggb)': 4389010944,
         'id(ggb.comm)': 4443888608,
         'id(ggb_comm_instance)': 4443888608}}


In [9]:
await ggb.function("getVersion")

'5.2.909.9'

In [18]:
# from ggblab.comm import report_comm_status
ggb.comm.status()

{'target_name': 'ggblab-comm',
 'registered': True,
 'target_comm_id': '69d931c2-e830-417e-90b1-3f2ec7db61aa',
 'has_oob_clients': False,
 'socketPath': None,
 'wsPort': 0}

In [13]:
def report_comm_status():
    """Helper to find the module-level `ggb_comm_instance` (if present)
    and return its status. Returns None if no instance is found.
    """
    try:
        ip = get_ipython()
        user_ns = getattr(ip, 'user_ns', {}) if ip is not None else {}
        inst = user_ns.get('ggb_comm_instance') or globals().get('ggb_comm_instance')
        if inst is None:
            return None
        try:
            return inst.status()
        except Exception:
            return None
    except Exception:
        return None

In [34]:
from IPython import get_ipython
from ggblab.comm import ggb_comm

inst = globals().get('ggb_comm_instance', None)
ggb_obj = globals().get('ggb', None)

# If inst and ggb exist and are different, make ggb use the registered instance:
if inst and ggb_obj and getattr(ggb_obj, "comm", None) is not inst:
    ggb_obj.comm = inst
    print("Repointed ggb.comm -> ggb_comm_instance")

Repointed ggb.comm -> ggb_comm_instance


In [15]:
ggb.comm.target_name

'ggblab-comm'

In [16]:
ggb.comm.target_comm

In [25]:
ggb.comm._registered

True

In [34]:
ggb.comm.recv_events.queue

deque([])

In [13]:
t = asyncio.ensure_future(ggb.init())
await t
await ggb.function("getAllObjectNames")

RuntimeError: No active Comm: GeoGebra().init() must be called in a notebook cell before sending commands.

In [8]:
_ggb

NameError: name '_ggb' is not defined

In [6]:
if getattr(ggb, 'comm', None) is not None:
    print(f"ggb.comm = {ggb.comm.target_comm}")
t = asyncio.ensure_future(ggb.init())
await asyncio.sleep(0)
await t
print(f"ggb.comm = {ggb.comm.target_comm}")
print(f"server_loop = {ggb.comm._server_loop}")
ggb.comm.debug = True
print(ggb.comm.logs)
# print(ggb.comm.recv_events.queue)
for i in range(10):
    print(ggb.comm.recv_events.queue)
    if not ggb.comm.recv_events.empty():
        break
    await asyncio.sleep(1)
print(f"ggb.comm = {ggb.comm.target_comm}")
print(f"server_loop = {ggb.comm._server_loop}")
print(ggb.comm.logs)
await ggb.function("getAllObjectNames")

Using local cached file: xsd/common.xsd
ggb.comm = None
server_loop = <_UnixSelectorEventLoop running=True closed=False debug=False>
[]
deque([])
deque([{'type': 'start', 'payload': {}}])
ggb.comm = None
server_loop = <_UnixSelectorEventLoop running=True closed=False debug=False>
['Clients connected: 1 (connects+=1, disconnects+=0)', 'Enqueuing event of type: start']


RuntimeError: No active Comm: GeoGebra().init() must be called in a notebook cell before sending commands.

In [8]:
ggb.comm.logs

['Clients connected: 1 (connects+=1, disconnects+=0)',
 'Enqueuing event of type: start',
 'No clients; waiting for client before sending 7a70f391-b798-4178-aa5e-6e0563a17454',
 'send(): has_clients=False, server_loop_present=True',
 'post_execute: event start',
 'post_execute: flushed 1 recv_events',
 'register_target_cb: <ipykernel.comm.comm.Comm object at 0x11dbe3d90>',
 '_pre_run_cell: registering Comm target ggblab-comm',
 'Registered post_execute handler for recv_events',
 '_pre_run_cell: registering Comm target ggblab-comm',
 'Registered post_execute handler for recv_events']

In [9]:
ggb.comm.target_comm

In [13]:
await ggb.function("getAllObjectNames")

[]

In [36]:
from IPython import get_ipython

ip = get_ipython()
cm = getattr(ip, 'kernel', None) and getattr(ip.kernel, 'comm_manager', None)

print("comm_manager available:", bool(cm))
if cm is None:
    raise RuntimeError("No kernel.comm_manager available in this context")

# List registered target names
print("registered targets:", list(cm.targets.keys()))

# Check specific target
target = 'ggblab-comm'
print(f"'{target}' registered:", target in cm.targets)

# If present, show the registered callback object
if target in cm.targets:
    print("handler repr:", repr(cm.targets[target]))

comm_manager available: True
registered targets: ['jupyter.widget', 'jupyter.widget.control', 'ggblab-comm']
'ggblab-comm' registered: True
handler repr: <bound method ggb_comm.register_target_cb of <ggblab.comm.ggb_comm object at 0x128e25950>>


In [45]:
await ggb.function("getVersion")

'5.2.909.9'

In [38]:
print("registered flag:", getattr(ggb_comm_instance, "_registered", None))
print("target_comm:", repr(getattr(ggb_comm_instance, "target_comm", None)))

registered flag: True
target_comm: <ipykernel.comm.comm.Comm object at 0x12b418d60>


In [10]:
from ggblab.comm import ggb_comm
inst = globals().get('ggb_comm_instance', None)
print("ggb_comm_instance exists:", bool(inst))
if not inst:
    inst = ggb_comm()
    print("Created ephemeral ggb_comm_instance (not persisted)")

print("registered flag:", getattr(inst, "_registered", None))
tc = getattr(inst, "target_comm", None)
print("target_comm repr:", repr(tc))
print("target_comm comm_id:", getattr(tc, "comm_id", getattr(tc, "commId", None)))
print("comm_manager targets:", list(get_ipython().kernel.comm_manager.targets.keys()))
print("ggb_comm_instance logs (last 10):", getattr(inst, "logs", [])[-10:])

ggb_comm_instance exists: True
registered flag: True
target_comm repr: <ipykernel.comm.comm.Comm object at 0x12223dbd0>
target_comm comm_id: dc94b518-69f7-426a-afea-6972e2068b27
comm_manager targets: ['jupyter.widget', 'jupyter.widget.control', 'ggblab-comm']
ggb_comm_instance logs (last 10): ['Clients connected: 1 (connects+=1, disconnects+=0)', 'post_execute: event start', 'post_execute: flushed 1 recv_events', 'No clients; waiting for client before sending dce44dee-f4d1-4ba4-82e4-ac0c763f9dc6']


In [11]:
ggb_comm_instance.debug = True
ggb_comm_instance.send('{"type":"debug","payload":"hello-from-kernel"}')

In [13]:
from IPython import get_ipython
from ggblab.comm import ggb_comm

inst = globals().get('ggb_comm_instance', None)
ggb_obj = globals().get('ggb', None)

print("ggb_comm_instance exists:", bool(inst), "id:", id(inst))
if inst:
    print("  _registered:", getattr(inst, "_registered", None))
    print("  target_comm repr:", repr(getattr(inst, "target_comm", None)))
    print("  target_comm.comm_id:", getattr(getattr(inst, "target_comm", None), "comm_id", None))

print("ggb object exists:", bool(ggb_obj), "id:", id(ggb_obj))
if ggb_obj:
    commobj = getattr(ggb_obj, "comm", None)
    print("  ggb.comm id:", id(commobj), "repr:", repr(commobj))
    print("  ggb.comm._registered:", getattr(commobj, "_registered", None))
    print("  ggb.comm.target_comm.comm_id:", getattr(getattr(commobj, "target_comm", None), "comm_id", None))

cm = get_ipython().kernel.comm_manager
print("comm_manager targets:", list(cm.targets.keys()))
for name, cb in cm.targets.items():
    print(name, "->", cb, "self id:", id(getattr(cb, '__self__', None)))

ggb_comm_instance exists: True id: 4867741712
  _registered: True
  target_comm repr: <ipykernel.comm.comm.Comm object at 0x12223dbd0>
  target_comm.comm_id: dc94b518-69f7-426a-afea-6972e2068b27
ggb object exists: True id: 4865934400
  ggb.comm id: 4865935072 repr: <ggblab.comm.ggb_comm object at 0x1220846e0>
  ggb.comm._registered: True
  ggb.comm.target_comm.comm_id: None
comm_manager targets: ['jupyter.widget', 'jupyter.widget.control', 'ggblab-comm']
jupyter.widget -> <function Widget.handle_comm_opened at 0x12014c9e0> self id: 4372411728
jupyter.widget.control -> <bound method Widget.handle_control_comm_opened of <class 'ipywidgets.widgets.widget.Widget'>> self id: 4607848400
ggblab-comm -> <bound method ggb_comm.register_target_cb of <ggblab.comm.ggb_comm object at 0x12223d810>> self id: 4867741712


In [16]:
from IPython import get_ipython
from ggblab.comm import ggb_comm

inst = globals().get('ggb_comm_instance', None)
ggb_obj = globals().get('ggb', None)

# If inst and ggb exist and are different, make ggb use the registered instance:
if inst and ggb_obj and getattr(ggb_obj, "comm", None) is not inst:
    ggb_obj.comm = inst
    print("Repointed ggb.comm -> ggb_comm_instance")

In [4]:
# case 1 of interactions: algebraic command
# [GeoGebra Manual :: GeoGebra Manual]
# (https://geogebra.github.io/docs/manual/)
r = await ggb.command("O = (0, 0)")
r

'O'

In [5]:
# case 2 of interactions: API function
# [GeoGebra Apps API :: GeoGebra Manual]
# (https://geogebra.github.io/docs/reference/en/GeoGebra_Apps_API/)
r = await ggb.function("getAllObjectNames")
r

['O']

In [6]:
r = await ggb.function("newConstruction")
r

In [7]:
# load from .ggb (zipped, and base64 encoded)
c = ggb.file.load('2025_06_08.ggb')

In [8]:
# sending loaded construction to GeoGebra view and draw
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [9]:
from itertools import zip_longest

l0 = range(10)
await ggb.function("setLayerVisible", list(zip_longest(list(l0), [], fillvalue=False)))
l0 = [9, 0]
await ggb.function("setLayerVisible", list(zip_longest(list(l0), [], fillvalue=True)))

[None, None]

In [10]:
# case 3 of interactions: XML attributes

In [9]:
r = await ggb.function("getXML", ['f'])
print(r)

<element type="segment" label="f">
	<show object="true" label="true"/>
	<objColor r="0" g="0" b="0" alpha="0"/>
	<layer val="0"/>
	<labelMode val="0"/>
	<decoration type="4"/>
	<lineStyle thickness="5" type="10" typeHidden="1" opacity="178"/>
	<eqnStyle style="implicit"/>
	<outlyingIntersections val="false"/>
	<keepTypeOnTransform val="true"/>
	<startStyle val="arrow"/>
	<endStyle val="line"/>
	<coords x="0.16000000000000014" y="3.34" z="-5.3124"/>
</element>



In [12]:
o2 = c.ggb_schema.decode(r)
o2

{'@type': 'point',
 '@label': 'A',
 'show': [{'@object': True, '@label': True, '@ev': 4}],
 'objColor': [{'@r': 176, '@g': 0, '@b': 32, '@alpha': 0.0}],
 'layer': [{'@val': 9}],
 'labelMode': [{'@val': 0}],
 'animation': [{'@step': '0.1', '@type': 1, '@playing': False}],
 'pointSize': [{'@val': 5.0}],
 'pointStyle': [{'@val': 0}],
 'coords': [{'@x': '10', '@y': '0', '@z': '1'}]}

In [13]:
o2['show'][0]['@object'] = False
o2

{'@type': 'point',
 '@label': 'A',
 'show': [{'@object': False, '@label': True, '@ev': 4}],
 'objColor': [{'@r': 176, '@g': 0, '@b': 32, '@alpha': 0.0}],
 'layer': [{'@val': 9}],
 'labelMode': [{'@val': 0}],
 'animation': [{'@step': '0.1', '@type': 1, '@playing': False}],
 'pointSize': [{'@val': 5.0}],
 'pointStyle': [{'@val': 0}],
 'coords': [{'@x': '10', '@y': '0', '@z': '1'}]}

In [14]:
# construct xml attributes from dict
x = xmlschema.etree_tostring(c.ggb_schema.encode(o2, 'element'))
print(x)

<element type="point" label="A">
    <show object="false" label="true" ev="4" />
    <objColor r="176" g="0" b="32" alpha="0.0" />
    <layer val="9" />
    <labelMode val="0" />
    <animation step="0.1" type="1" playing="false" />
    <pointSize val="5.0" />
    <pointStyle val="0" />
    <coords x="10" y="0" z="1" />
</element>


In [15]:
# update the applet
r = await ggb.function("evalXML", [x])

In [16]:
r = await ggb.function("getBase64")

In [17]:
type(r), type(r.encode('ascii'))

(str, bytes)

In [18]:
# encode str to bytes for save
ggb.construction.base64_buffer = r.encode('ascii')
# ggb.construction.base64_buffer

In [19]:
# using API, some part of archive would lost...
c.save()